# HONOR 200 Pro — Full Phone Backup
**Destination:** `E:\HonorPhoneBackup22MAY2026`

**Prerequisites:** Phone connected via USB, MTP/File Transfer mode active.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import win32com.client
import os
import time
from datetime import datetime
from IPython.display import clear_output

DEST = r"E:\HonorPhoneBackup22MAY2026"
os.makedirs(DEST, exist_ok=True)

print(f"Destination : {DEST}")
print(f"Started     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Ready.")

In [ ]:
# ── Cell 2: Connect to phone & verify Internal Storage ────────────────────────
shell = win32com.client.Dispatch("Shell.Application")
computer = shell.Namespace(0x11)  # This PC / Computer

phone_item = None
for item in computer.Items():
    if "HONOR" in item.Name:
        phone_item = item
        print(f"Device found : {item.Name}")
        break

if phone_item is None:
    raise RuntimeError("HONOR phone not found. Make sure it is connected and USB mode is set to File Transfer.")

# Navigate to Internal Storage using GetFolder (works for MTP devices)
storage_item = None
for item in phone_item.GetFolder.Items():
    if item.Name == "Internal storage":
        storage_item = item
        break

if storage_item is None:
    raise RuntimeError("'Internal storage' not found on the device.")

storage_folder = storage_item.GetFolder
top_items = list(storage_folder.Items())
folders = [i.Name for i in top_items if i.IsFolder]
loose   = [i.Name for i in top_items if not i.IsFolder]

print(f"\nInternal storage — {len(folders)} folders, {len(loose)} loose files")
print("\nFolders:")
for f in sorted(folders):
    print(f"  {f}")
if loose:
    print("\nLoose files at root:")
    for f in sorted(loose):
        print(f"  {f}")

In [ ]:
# ── Cell 3: Run the backup ────────────────────────────────────────────────────
# CopyHere flags:
#   4    = FOF_SILENT        (no progress dialog)
#   16   = FOF_NOCONFIRMATION (no overwrite prompts)
#   512  = FOF_NOCONFIRMMKDIR (auto-create dirs)
#   1024 = FOF_NOERRORUI     (suppress error dialogs)
COPY_FLAGS = 4 | 16 | 512 | 1024

dest_ns = shell.Namespace(DEST)
if dest_ns is None:
    raise RuntimeError(f"Cannot access destination folder: {DEST}")

print("Launching backup (Windows Shell is copying in the background)...")
print("Monitor cell below will track progress.\n")

backup_start = time.time()
dest_ns.CopyHere(storage_item, COPY_FLAGS)

# ── Progress monitor ──────────────────────────────────────────────────────────
STABLE_THRESHOLD = 12   # stop after 60 s with no new files
CHECK_INTERVAL   = 5    # seconds between checks

prev_count  = -1
stable_runs = 0

def dest_stats():
    count = 0
    size  = 0
    for root, _, files in os.walk(DEST):
        for f in files:
            try:
                size += os.path.getsize(os.path.join(root, f))
                count += 1
            except OSError:
                pass
    return count, size

try:
    while stable_runs < STABLE_THRESHOLD:
        time.sleep(CHECK_INTERVAL)
        count, size = dest_stats()
        elapsed = time.time() - backup_start
        clear_output(wait=True)
        print(f"Files copied : {count:,}")
        print(f"Size so far  : {size / 1024**3:.3f} GB")
        print(f"Elapsed      : {elapsed/60:.1f} min")
        print(f"Stability    : {stable_runs}/{STABLE_THRESHOLD} (stop when stable)")
        if count == prev_count:
            stable_runs += 1
        else:
            stable_runs = 0
        prev_count = count
    print("\nBackup appears complete (no new files for 60 s).")
except KeyboardInterrupt:
    print("\nMonitoring stopped. Copy may still be running in the background.")

In [ ]:
# ── Cell 4: Verify backup ─────────────────────────────────────────────────────
count, size_bytes = dest_stats()
size_gb = size_bytes / 1024**3

# Count destination sub-folders (top level)
top_dirs = [d for d in os.listdir(DEST) if os.path.isdir(os.path.join(DEST, d))]

print("=" * 45)
print("  BACKUP COMPLETE")
print("=" * 45)
print(f"  Destination : {DEST}")
print(f"  Folders     : {len(top_dirs)}")
print(f"  Files       : {count:,}")
print(f"  Size        : {size_gb:.2f} GB")
print(f"  Finished    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 45)